# Đề tài: SmartHVACControlByDoubleQLearning
## Notebook 01: Phân tích và Khám phá Dữ liệu CU-BEMS (Exploratory Data Analysis - EDA)

**Mục tiêu:**
- Khám phá bộ dữ liệu cảm biến tòa nhà thông minh CU-BEMS (Chulalongkorn University Building Energy Management System).
- Phân tích các thông số nhiệt độ (`degC`), độ ẩm (`RH%`), độ rọi ánh sáng (`lux`), công suất chiếu sáng (`Light kW`), công suất phụ tải (`Plug kW`), và công suất điều hòa không khí (`HVAC / AC kW`).
- Phân tích chu kỳ tiêu thụ năng lượng theo giờ trong ngày (Diurnal Pattern) và ngày làm việc so với cuối tuần.
- Đánh giá tương quan nhiệt - điện làm cơ sở xây dựng mô hình môi trường Reinforcement Learning.

In [ ]:
import sys
from pathlib import Path
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# Cấu hình đường dẫn thư viện của dự án
project_root = Path('..').resolve()
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

from src.data.loader import load_raw_floor_data, preprocess_cubems_data
from src.util.visualization import (
    plot_time_series,
    plot_correlation_matrix,
    plot_diurnal_patterns,
    plot_comfort_distribution
)

plt.style.use('seaborn-v0_8-whitegrid' if 'seaborn-v0_8-whitegrid' in plt.style.available else 'default')

### 1. Tải và kiểm tra dữ liệu đã qua tiền xử lý

In [ ]:
processed_path = project_root / 'data' / 'processed' / 'cubems_zone1_hourly.csv'
df = pd.read_csv(processed_path)
df['datetime'] = pd.to_datetime(df['datetime'])
print(f'Tổng số mẫu theo giờ: {len(df)}')
print(f'Khoảng thời gian: {df["datetime"].min()} đến {df["datetime"].max()}')
df.head()

### 2. Thống kê mô tả các chỉ số môi trường & phụ tải

In [ ]:
stats_cols = ['temp_c', 'humidity_rh', 'light_lux', 'plug_kw', 'light_kw', 'internal_heat_kw', 'hvac_kw']
df[stats_cols].describe().round(2)

### 3. Trực quan hóa diễn biến chuỗi thời gian (Time-Series Profile)
Xem diễn biến 1 tuần tiêu biểu của các thông số Nhiệt độ, Độ ẩm, Chiếu sáng, Phụ tải ổ cắm và Công suất HVAC.

In [ ]:
# Vẽ và lưu biểu đồ chuỗi thời gian 168 giờ
plot_time_series(df, start_idx=200, n_hours=168)
from IPython.display import Image
Image(filename=str(project_root / 'result' / 'figure' / 'eda_time_series.png'))

### 4. Ma trận tương quan giữa các biến (Correlation Matrix)
Nhận diện mức độ tác động của công suất làm mát HVAC và nhiệt lượng nội tại lên nhiệt độ trong phòng.

In [ ]:
plot_correlation_matrix(df)
Image(filename=str(project_root / 'result' / 'figure' / 'eda_correlation_matrix.png'))

### 5. Phân tích chu kỳ ngày/đêm (Diurnal Patterns: Ngày làm việc vs Cuối tuần)
So sánh nhiệt độ và phụ tải điện giữa ngày làm việc (có người hoạt động) và cuối tuần (trống phòng).

In [ ]:
plot_diurnal_patterns(df)
Image(filename=str(project_root / 'result' / 'figure' / 'eda_diurnal_patterns.png'))

### 6. Phân bố nhiệt độ phòng so với Tiêu chuẩn Tiện nghi Nhiệt ASHRAE 55

In [ ]:
plot_comfort_distribution(df)
Image(filename=str(project_root / 'result' / 'figure' / 'eda_comfort_distribution.png'))

### 7. Nhận xét & Kết luận từ Dữ liệu CU-BEMS:
1. **Mối quan hệ nhiệt - công suất:** Tương quan giữa công suất điều hòa `hvac_kw` và nhiệt độ phòng `temp_c` là xấp xỉ `-0.81`, khẳng định hệ thống điều hòa là tác nhân chi phối trực tiếp tốc độ giảm nhiệt trong tòa nhà.
2. **Tải nhiệt nội tại (Internal Heat Gain):** Phụ tải máy móc (`plug_kw`) và chiếu sáng (`light_kw`) đạt đỉnh trong khung giờ 08:00 - 17:00 các ngày trong tuần (lên đến ~12 kW), tạo ra nhiệt lượng đáng kể đòi hỏi HVAC phải tăng công suất làm mát.
3. **Tiện nghi nhiệt:** Trong dữ liệu thực tế, nhiệt độ thường dao động mạnh từ 18°C (làm lạnh quá mức vào ban ngày) đến hơn 30°C (khi tắt HVAC vào ban đêm). Điều này chứng minh tiềm năng rất lớn của bộ điều khiển thông minh để ổn định nhiệt độ trong dải 22°C - 24.5°C đồng thời tránh lãng phí năng lượng.